# Initial Set up

In [2]:
import os
import import_ipynb
from mimic_utils_text import InHospitalMortalityReader, read_chunk
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

importing Jupyter notebook from mimic_utils_text.ipynb


Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.


In [3]:
padding_num_features = 35
np.random.seed(42)

## Set logger

In [4]:
import logging

LOGGER = logging.getLogger("LSTM")

LOGGER.setLevel(logging.INFO)
logging_format = logging.Formatter(
    "[%(asctime)s - %(filename)s:%(lineno)s - %(funcName)s() - %(levelname)s %(message)s"
)
ch = logging.StreamHandler()
ch.setFormatter(logging_format)
LOGGER.addHandler(ch)

In [5]:
# pad missing values in the nested lists with 0s
def pad_missing_value_with_zero(data):
    a1 = np.zeros((len(data), max([len(k) for k in data]), padding_num_features))  # 35
    for ctr, k in enumerate(data):
        # print(ctr, len(k), k)
        # Convert string representations of Boolean values to numerical values
        k = np.array([[1.0 if val == "True" else 0.0 if val == "False" else float(val) for val in row] for row in k])

        a1[ctr, : len(k), :] = k
    return a1

In [6]:
# pytroch class for reading data into batches
class MIMICDataset(Dataset):
    """
    Loads time series data into memory from a text file,
    split by newlines.
    """

    def __init__(self, reader, target_repl=False, batch_labels=False):
        self.data = []
        self.y = []
        N = reader.get_number_of_examples()
        print(f"Number of examples:{N}")
        # read data form cvs files
        ret = read_chunk(reader, N)
        # read into memory structured data X and labels y
        # print(ret)

        data = ret["X"]
        ts = ret["t"]
        labels = ret["y"]
        names = ret["name"]
        self.features = ret["header"]

        # print(data)

        # pad missing values in the list of arrays with 0s
        data = pad_missing_value_with_zero(data)

        self.data = np.array(data, dtype=np.float32)
        self.T = self.data.shape[1]

        if batch_labels:
            self.y = np.array([[l] for l in labels], dtype=np.float32)
        else:
            self.y = np.array(labels, dtype=np.float32)
        if target_repl:
            self.y = self._extend_labels(self.y)

    def _extend_labels(self, labels):
        # (B,)
        labels = labels.repeat(self.T, axis=1)  # (B, T)
        return labels

    def __len__(self):
        # overide len to get number of instances
        return len(self.data)

    def __getitem__(self, idx):
        # get features (physiological variables x) and label for a given instance index
        return self.data[idx], self.y[idx]

In [7]:
data_dir = "data/AKI/fts_extract_race_groups"

In [8]:
model_path = 'data/models/2024-12-13/fts_extract_race_groups/_dropout_0.2,batch_size_64,lr_0.pth'

args = {
    "best_model": model_path,
    "dim": 35,
    "dropout": 0.2,
    "batch_size": 16,
    "emb_size": 35,
    "aggregation_type": "mean",
    "bidirectional": False,
    "data": data_dir,  # path to data
    "notes": data_dir,  # the code ignores the text
    "timestep": 1.0,
    "imputation": "previous",
    "normalizer_state": None,
}

# Load data

In [ ]:
# Load training data
train_reader = InHospitalMortalityReader(
    dataset_dir=os.path.join(args['data'], "train"),
    notes_dir=args['notes'],
    listfile=os.path.join(args['notes'], "train_listfile.csv"),
    period_length=48.0,
)

train_dataset = MIMICDataset(train_reader, batch_labels=True)
train_dl = DataLoader(train_dataset, batch_size=100, shuffle=False)

In [9]:
test_reader = InHospitalMortalityReader(
    dataset_dir=os.path.join(args['data'], "test"),
    notes_dir=args['notes'],
    listfile=os.path.join(args['notes'], "test_listfile.csv"),
    period_length=48.0,
)

test_dataset = MIMICDataset(test_reader, batch_labels=True)
test_dl = DataLoader(test_dataset, batch_size=100, shuffle=False)
# [B, M, feat_size]
feat_size = test_dataset.data.shape[-1]

InHospitalMortalityReader init completed
Number of examples:6368
Reading chunk of size 6368
Number of records with more than 48 hours: 56


In [10]:
test_dataset.data.shape

(6368, 96, 35)

# Load Model

In [11]:
import torch.nn as nn
import torch

In [12]:
# model
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        tag_size,
        hidden_size,
        feat_size,
        emb_size,
        bidirectional=False,
        dropout=0.2,
        aggregation_type="last_state",
    ):
        """
        constructor, here we define the hidden layers for our architecture
        """
        super().__init__()

        # define if the rnn will be bidirectional
        self.bidirectional = bidirectional

        # define the aggregation type of the features for the classifier for example, you can take the mean
        self.aggregation_type = aggregation_type
        self.encoder = nn.Linear(feat_size, emb_size, bias=True)
        
        # Create a (bidirectional) LSTM to encode sequence
        self.lstm = nn.LSTM(emb_size, hidden_size, batch_first=True, bidirectional=bidirectional)

        # The output of the LSTM doubles if we use a bidirectional encoder.
        encoding_size = hidden_size * 2 if bidirectional else hidden_size
        self.combination_layer = nn.Linear(encoding_size, encoding_size)

        # Create affine layer to project to the classes
        self.projection = nn.Linear(encoding_size, tag_size)
        
        # dropout layer for regularizetion of a sequence
        self.dropout_layer = nn.Dropout(p=dropout)
        self.relu = nn.ReLU()

    def forward(self, x, seq_mask=None, seq_len=None):
        # input size
        # [B, T, feat_size] batch, time and features
        # return unormalized probabilities (logits)
        # the loss will compute the sigmoid and negative log-likelihood
        # output size
        # [B, num_class] batch, and 1 class

        # [B, T, F] batch, time, features
        h1 = self.encoder(x)
        h1 = self.relu(h1)
        # [B, T, H] batch, time, hidden or hidden * 2
        outputs, (final, _) = self.lstm(h1)

        if self.aggregation_type == "mean":
            # mean over hidden states of LSTM
            outputs = self.dropout_layer(outputs)
            h = self.relu(self.combination_layer(outputs))
            # [B, H] batch, hidden
            h = h.mean(dim=1)  # mean over time dimension
        elif self.aggregation_type == "last_state":
            # last hidden state of the lstm or concat of bidirectional forward and backward states
            if self.bidirectional:
                h_T_fwd = final[0]  # lstm 1, last hidden state of forward lstm
                h_T_bwd = final[1]  # lstm 2. last hidden state of backward lstm
                # [B, H*2]
                h = torch.cat(
                    [h_T_fwd, h_T_bwd], dim=-1
                )  # concatenate the forward with the backward in the last dimension (feat)
            else:
                h = final[-1]
            h = self.relu(self.combination_layer(h))
            h = self.dropout_layer(h)
        # [B, H] # summary for each patient
        # [B, 1]
        logits = self.projection(h)

        return logits

In [13]:
# Define the classification model.
model = LSTMClassifier(
    tag_size=1,  # binary
    feat_size=feat_size,
    hidden_size=args["dim"],
    emb_size=args["emb_size"],
    bidirectional=args["bidirectional"],
    dropout=args["dropout"],
    aggregation_type=args["aggregation_type"],
)

# load trained model from file
model.load_state_dict(torch.load(args["best_model"]))
LOGGER.info(model)

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
model = model.to(device)

[2025-02-02 23:44:36,920 - 225444385.py:14 - <module>() - INFO LSTMClassifier(
  (encoder): Linear(in_features=35, out_features=35, bias=True)
  (lstm): LSTM(35, 35, batch_first=True)
  (combination_layer): Linear(in_features=35, out_features=35, bias=True)
  (projection): Linear(in_features=35, out_features=1, bias=True)
  (dropout_layer): Dropout(p=0.2, inplace=False)
  (relu): ReLU()
)


# TimeSHAP

In [168]:
!pip install timeshap

In [14]:
from timeshap import __version__
__version__

'1.0.4'

In [21]:
test_dataset.data
df_testdata = pd.DataFrame(test_dataset.data.reshape(-1, test_dataset.data.shape[-1]), columns=test_dataset.features)
df_testdata['instance'] = np.repeat(np.arange(len(test_dataset)), test_dataset.data.shape[1])
df_testdata

,Hours,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,hematocrit_avg,...,white_group,black_african_group,asian_group,hispanic_group,unknown_group,native_group,other_group,ELECTIVE,URGENT,instance
0,0.466667,0.210526,0.346154,0.460784,1.0,0.107143,0.661972,0.120715,0.304878,0.462094,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,1.133333,0.210526,0.346154,0.460784,1.0,0.107143,0.563380,0.120715,0.329268,0.462094,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2.133333,0.210526,0.346154,0.460784,1.0,0.107143,0.366197,0.120715,0.463415,0.462094,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,3.133333,0.210526,0.346154,0.460784,1.0,0.107143,0.450704,0.120715,0.280488,0.462094,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,4.133333,0.210526,0.346154,0.460784,1.0,0.107143,0.239437,0.120715,0.304878,0.462094,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
611323,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6367
611324,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6367
611325,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6367
611326,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6367


In [22]:
from timeshap.wrappers import TorchModelWrapper
model_wrapped = TorchModelWrapper(model)
f_hs = lambda x, y=None: model_wrapped.predict_last_hs(x, y)

In [25]:
df_testdata.columns.to_list()

['Hours',
 'aniongap_avg',
 'bicarbonate_avg',
 'bun_avg',
 'chloride_avg',
 'creat',
 'diasbp_mean',
 'glucose_avg',
 'heartrate_mean',
 'hematocrit_avg',
 'hemoglobin_avg',
 'potassium_avg',
 'resprate_mean',
 'sodium_avg',
 'spo2_mean',
 'sysbp_mean',
 'uo_rt_12hr',
 'uo_rt_24hr',
 'uo_rt_6hr',
 'wbc_avg',
 'sedative',
 'vasopressor',
 'vent',
 'anchor_age',
 'F',
 'M',
 'white_group',
 'black_african_group',
 'asian_group',
 'hispanic_group',
 'unknown_group',
 'native_group',
 'other_group',
 'ELECTIVE',
 'URGENT',
 'instance']

In [29]:
from timeshap.utils import calc_avg_event
average_event = calc_avg_event(df_testdata, numerical_feats=test_dataset.features, categorical_feats=[])

# Local report

In [30]:
id_instance = 0
single_instance = df_testdata[df_testdata['instance'] == id_instance]

In [33]:
feat_dict = {feat: feat for feat in test_dataset.features}
feat_dict

{'Hours': 'Hours',
 'aniongap_avg': 'aniongap_avg',
 'bicarbonate_avg': 'bicarbonate_avg',
 'bun_avg': 'bun_avg',
 'chloride_avg': 'chloride_avg',
 'creat': 'creat',
 'diasbp_mean': 'diasbp_mean',
 'glucose_avg': 'glucose_avg',
 'heartrate_mean': 'heartrate_mean',
 'hematocrit_avg': 'hematocrit_avg',
 'hemoglobin_avg': 'hemoglobin_avg',
 'potassium_avg': 'potassium_avg',
 'resprate_mean': 'resprate_mean',
 'sodium_avg': 'sodium_avg',
 'spo2_mean': 'spo2_mean',
 'sysbp_mean': 'sysbp_mean',
 'uo_rt_12hr': 'uo_rt_12hr',
 'uo_rt_24hr': 'uo_rt_24hr',
 'uo_rt_6hr': 'uo_rt_6hr',
 'wbc_avg': 'wbc_avg',
 'sedative': 'sedative',
 'vasopressor': 'vasopressor',
 'vent': 'vent',
 'anchor_age': 'anchor_age',
 'F': 'F',
 'M': 'M',
 'white_group': 'white_group',
 'black_african_group': 'black_african_group',
 'asian_group': 'asian_group',
 'hispanic_group': 'hispanic_group',
 'unknown_group': 'unknown_group',
 'native_group': 'native_group',
 'other_group': 'other_group',
 'ELECTIVE': 'ELECTIVE',
 'UR

In [35]:
from timeshap.explainer import local_report

pruning_dict = {'tol': 0.025}
event_dict = {'rs': 42, 'nsamples': 32000}
feature_dict = {'rs': 42, 'nsamples': 32000, 'feature_names': test_dataset.features, 'plot_features': feat_dict} #, 'plot_features': plot_feats
cell_dict = {'rs': 42, 'nsamples': 32000, 'top_x_feats': 2, 'top_x_events': 2}
local_report(f_hs, single_instance, pruning_dict, event_dict, feature_dict, cell_dict, average_event, test_dataset.features, entity_col ='instance', entity_uuid=id_instance)

the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype para

alt.HConcatChart(...)

In [1]:
from shap import __version__
__version__

'0.37.0'

In [172]:
!pip install shap==0.37.0

     ---------------------------------------- 0.0/326.5 kB ? eta -:--:--
     - -------------------------------------- 10.2/326.5 kB ? eta -:--:--
     ---- -------------------------------- 41.0/326.5 kB 388.9 kB/s eta 0:00:01
     -------------------------------------- 326.5/326.5 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for shap: filename=shap-0.37.0-cp311-cp311-win_amd64.whl size=376469 sha256=a3017a5665ce786dd5984eaa28b5e3f97336d8a571c5fd3b7ff245879f0f1905
  Stored in directory: c:\users\daima researcher\appdata\local\pip\cache\wheels\e6\b9\4a\ab17a17d2aa019318c017e406b9015fb96aaf7d73a0a96f17f
Successfully built shap
  Attempting uninstall: slicer
    Found existing installation: slicer 0.0.8
    Uninstalling slicer-0.0.8:
      Successfully uninstalled slicer-0.0.8
  Attempting uninstall: shap
    Found existing installation: shap 0.46.0
    Uninstalling shap-0.46.0:
      Successf

  You can safely remove it manually.
